In [188]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Librerias

In [189]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import RMSprop, Adam, SGD, Nadam, AdamW
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import tensorflow as tf

import json

import pickle

from tensorflow.keras.models import load_model

from itertools import combinations



#Ruta para guardar/importar archivos

In [190]:
ruta='/content/drive/MyDrive/Colab Notebooks/Tesis/Codigos/'

#Preprocesamiento de la base

In [191]:
#@title Funcion
def preparar_pipeline(vars_keep, input_length, output_length,var_n):

    import numpy as np
    from sklearn.preprocessing import MinMaxScaler

    col_ref = vars_keep.index(var_n)

    for c in countries:
      name = c.replace(" ", "_")
      globals()[f"base_{name}"] = df[df["country"] == c]

    for name in globals():
      if name.startswith("base_"):
          globals()[name] = globals()[name].sort_values(by="date")

    # Años
    ano_ini_entre = 1983
    ano_fin_entre = 2010
    ano_ini_val = 2011
    ano_fin_val = 2016
    ano_ini_test = 2017
    ano_fin_test = 2022

    anos_entre = ano_fin_entre - ano_ini_entre + 1
    anos_vali = ano_fin_val - ano_ini_val + 1
    anos_test = ano_fin_test - ano_ini_test + 1

    # Selección de columnas
    for name in globals():
        if name.startswith("base_"):
            globals()[name] = globals()[name][vars_keep]

    # ---------------- TRAIN ----------------
    for name in list(globals().keys()):
        if name.startswith("base_"):

            x_train, y_train = [], []
            array = globals()[name].to_numpy()

            for i in range(anos_entre - input_length - output_length + 1):

                x_train.append(array[i:i+input_length])

                y_train.append(
                    array[i+input_length:i+input_length+output_length, -1]
                    .reshape(output_length, 1)
                )

            clean = name.replace("base_", "")

            globals()[f"x_train_{clean}"] = np.array(x_train)
            globals()[f"y_train_{clean}"] = np.array(y_train)

    # ---------------- VALID ----------------
    for name in list(globals().keys()):
        if name.startswith("base_"):

            x_val, y_val = [], []
            array = globals()[name].to_numpy()

            for i in range(
                anos_entre - input_length,
                anos_entre + anos_vali - output_length - input_length + 1
            ):

                x_val.append(array[i:i+input_length])

                y_val.append(
                    array[i+input_length:i+input_length+output_length, -1]
                    .reshape(output_length, 1)
                )

            clean = name.replace("base_", "")

            globals()[f"x_val_{clean}"] = np.array(x_val)
            globals()[f"y_val_{clean}"] = np.array(y_val)

    # ---------------- TEST ----------------
    for name in list(globals().keys()):
        if name.startswith("base_"):

            x_test, y_test = [], []
            array = globals()[name].to_numpy()

            for i in range(
                anos_entre + anos_vali - input_length,
                anos_entre + anos_vali + anos_test - output_length - input_length + 1
            ):

                x_test.append(array[i:i+input_length])

                y_test.append(
                    array[i+input_length:i+input_length+output_length, -1]
                    .reshape(output_length, 1)
                )

            clean = name.replace("base_", "")

            globals()[f"x_test_{clean}"] = np.array(x_test)
            globals()[f"y_test_{clean}"] = np.array(y_test)

    # ---------------- SCALERS ----------------
    for name in list(globals().keys()):
        if name.startswith("base_"):
            clean = name.replace("base_", "")
            globals()[f"scaler_{clean}"] = [
                MinMaxScaler(feature_range=(-1, 1))
                for _ in range(len(vars_keep))
            ]

    # ---------------- CREAR ARRAYS VACÍOS ----------------
    for name in list(globals().keys()):

        if name.startswith("x_train"):
            clean = name.replace("x_train", "")
            globals()[f"x_tr{clean}"] = np.zeros(globals()[name].shape)

        if name.startswith("y_train"):
            clean = name.replace("y_train", "")
            globals()[f"y_tr{clean}"] = np.zeros(globals()[name].shape)

        if name.startswith("x_val"):
            clean = name.replace("x_val", "")
            globals()[f"x_vl{clean}"] = np.zeros(globals()[name].shape)

        if name.startswith("y_val"):
            clean = name.replace("y_val", "")
            globals()[f"y_vl{clean}"] = np.zeros(globals()[name].shape)

        if name.startswith("x_test"):
            clean = name.replace("x_test", "")
            globals()[f"x_tst{clean}"] = np.zeros(globals()[name].shape)

        if name.startswith("y_test"):
            clean = name.replace("y_test", "")
            globals()[f"y_tst{clean}"] = np.zeros(globals()[name].shape)

    # ---------------- ESCALADO X TRAIN ----------------
    for name in list(globals().keys()):
        if name.startswith("x_train_"):

            clean = name.replace("x_train_", "")
            X = globals()[name]
            scalers = globals()[f"scaler_{clean}"]

            for i in range(len(vars_keep)):

                data = X[:, :, i].reshape(-1, 1)
                scaled = scalers[i].fit_transform(data)

                globals()[f"x_tr_{clean}"][:, :, i] = scaled.reshape(
                    X[:, :, i].shape
                )

    # ---------------- ESCALADO X VALID ----------------
    for name in list(globals().keys()):
        if name.startswith("x_val_"):

            clean = name.replace("x_val_", "")
            X = globals()[name]
            scalers = globals()[f"scaler_{clean}"]

            for i in range(len(vars_keep)):

                data = X[:, :, i].reshape(-1, 1)
                scaled = scalers[i].transform(data)

                globals()[f"x_vl_{clean}"][:, :, i] = scaled.reshape(
                    X[:, :, i].shape
                )

    # ---------------- ESCALADO X TEST ----------------
    for name in list(globals().keys()):
        if name.startswith("x_test_"):

            clean = name.replace("x_test_", "")
            X = globals()[name]
            scalers = globals()[f"scaler_{clean}"]

            for i in range(len(vars_keep)):

                data = X[:, :, i].reshape(-1, 1)
                scaled = scalers[i].transform(data)

                globals()[f"x_tst_{clean}"][:, :, i] = scaled.reshape(
                    X[:, :, i].shape
                )

    # ---------------- ESCALADO Y ----------------
    for name in list(globals().keys()):
        if name.startswith("y_train_"):

            clean = name.replace("y_train_", "")
            X = globals()[name]
            scalers = globals()[f"scaler_{clean}"]

            data = X[:, :, 0].reshape(-1, 1)
            scaled = scalers[col_ref].transform(data)

            globals()[f"y_tr_{clean}"][:, :, 0] = scaled.reshape(X[:, :, 0].shape)

    for name in list(globals().keys()):
        if name.startswith("y_val_"):

            clean = name.replace("y_val_", "")
            X = globals()[name]
            scalers = globals()[f"scaler_{clean}"]

            data = X[:, :, 0].reshape(-1, 1)
            scaled = scalers[col_ref].transform(data)

            globals()[f"y_vl_{clean}"][:, :, 0] = scaled.reshape(X[:, :, 0].shape)

    for name in list(globals().keys()):
        if name.startswith("y_test_"):

            clean = name.replace("y_test_", "")
            X = globals()[name]
            scalers = globals()[f"scaler_{clean}"]

            data = X[:, :, 0].reshape(-1, 1)
            scaled = scalers[col_ref].transform(data)

            globals()[f"y_tst_{clean}"][:, :, 0] = scaled.reshape(X[:, :, 0].shape)

    # ---------------- BASES TOTALES ----------------
    x_total_train = np.empty((0, globals()["x_tr_Australia"].shape[1], globals()["x_tr_Australia"].shape[2]))

    for name in list(globals().keys()):
        if name.startswith("x_tr_"):
            x_total_train = np.concatenate((x_total_train, globals()[name]), axis=0)

    y_total_train = np.empty((0, globals()["y_tr_Australia"].shape[1], globals()["y_tr_Australia"].shape[2]))

    for name in list(globals().keys()):
        if name.startswith("y_tr_"):
            y_total_train = np.concatenate((y_total_train, globals()[name]), axis=0)

    x_total_valid = np.empty((0, globals()["x_vl_Australia"].shape[1], globals()["x_vl_Australia"].shape[2]))

    for name in list(globals().keys()):
        if name.startswith("x_vl_"):
            x_total_valid = np.concatenate((x_total_valid, globals()[name]), axis=0)

    y_total_valid = np.empty((0, globals()["y_vl_Australia"].shape[1], globals()["y_vl_Australia"].shape[2]))

    for name in list(globals().keys()):
        if name.startswith("y_vl_"):
            y_total_valid = np.concatenate((y_total_valid, globals()[name]), axis=0)

    x_total_test = np.empty((0, globals()["x_tst_Australia"].shape[1], globals()["x_tst_Australia"].shape[2]))

    for name in list(globals().keys()):
        if name.startswith("x_tst_"):
            x_total_test = np.concatenate((x_total_test, globals()[name]), axis=0)

    y_total_test = np.empty((0, globals()["y_tst_Australia"].shape[1], globals()["y_tst_Australia"].shape[2]))

    for name in list(globals().keys()):
        if name.startswith("y_tst_"):
            y_total_test = np.concatenate((y_total_test, globals()[name]), axis=0)

    return (
        x_total_train,
        y_total_train,
        x_total_valid,
        y_total_valid,
        x_total_test,
        y_total_test
    )

#Predicciones y Grafica

In [192]:
#@title Funcion
def prediccion_grafico_pais(
    modelo,
    df,
    country,
    year,
    vars_keep,
    input_length,
    output_length
):


    # ---------------- INPUT PARA PREDICCION ----------------
    df_sub = df[
        (df["country"] == country) &
        (df["date"] <= year) &
        (df["date"] > year - input_length)
    ]

    df_sub = df_sub.sort_values("date")

    x_array = df_sub[vars_keep].values

    x_array_scaled = []

    nn_country = country.replace(" ", "_")

    for name in list(globals().keys()):
        if name.startswith(f"scaler_{nn_country}"):

            scalers = globals()[name]

            for i in range(len(vars_keep)):
                data = x_array[:, i].reshape(-1, 1)
                scaled = scalers[i].transform(data)
                x_array_scaled.append(scaled)

    x_array_scaled = np.array(x_array_scaled).T

    # ---------------- PREDICCION ----------------
    y_pr_a = modelo.predict(x_array_scaled)

    # ---------------- DESESCALAR ----------------
    for name in list(globals().keys()):
        if name.startswith(f"scaler_{nn_country}"):
            y_pr_a = globals()[name][-1].inverse_transform(y_pr_a)

    y_pred = y_pr_a.flatten()

    # ---------------- DATOS HISTORICOS ----------------
    df_sub = df[
        (df["country"] == country) &
        (df["date"] <= year + output_length) &
        (df["date"] > year - input_length)
    ]

    df_sub = df_sub.sort_values("date")

    x_hist = df_sub["date"].values
    y_hist = df_sub["GDP_cnstn_usd"].values

    # ---------------- EJE PREDICCION ----------------
    x_pred = []

    for i in range(1, output_length + 1):
        x_pred.append(year + i)

    x_pred = np.array(x_pred)

    # Último histórico
    x_last = x_hist[-(output_length + 1)]
    y_last = y_hist[-(output_length + 1)]

    # Conectar histórico con predicción
    x_pred_full = np.insert(x_pred, 0, x_last)
    y_pred_full = np.insert(y_pred, 0, y_last)

    # ---------------- GRAFICO ----------------
    fig = plt.figure(figsize=(8,5))

    plt.plot(x_hist, y_hist, 'o-', color='blue', label='Histórico')

    plt.plot(
        x_pred_full,
        y_pred_full,
        's--',
        color='red',
        label='Predicción'
    )

    plt.plot(
        x_last,
        y_last,
        'o',
        color='green',
        markersize=8,
        label='Último histórico'
    )

    plt.xlabel("Año")
    plt.ylabel("GDP (const USD)")
    plt.title(f"GDP de {country}: Histórico vs Predicción")

    plt.legend()
    plt.grid(True)

    plt.show()

    #Error rmse
    rmse = np.sqrt(np.mean((y_pred - y_hist[-output_length:])**2))
    print(f"RMSE: {rmse}")

    return y_pred, fig

#Importar base de datos

In [193]:
df=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Tesis/base_datos.csv')
df

,country,date,GDP_cnstn,GDP_growth,GDP_cnstn_usd,GDP_pp_usd,GDP_pp_LCU,Inflatn,forgn_invst,forgn_invst_gdp,Fin_consum,Fin_consum_govern,Ext_balnce_g_s,Fin_consum_hous,Unmploy,Labor_part,GDP_usd
0,Australia,2022,2.552248e+12,4.253046,1.592358e+12,61200.458896,98092.754060,7.199544,6.870942e+10,4.052153,71.071701,22.017954,5.228418,49.053747,3.728,66.846,1.695628e+12
1,Australia,2021,2.448128e+12,2.006948,1.527397e+12,59465.535671,95312.000446,3.060670,3.145306e+10,2.015424,73.318371,22.366947,3.801393,50.951424,5.022,66.056,1.560617e+12
2,Australia,2020,2.399962e+12,-0.133532,1.497346e+12,58377.767052,93568.513198,1.865834,1.797684e+10,1.348260,74.278610,21.813861,3.376511,52.464749,6.394,64.980,1.333336e+12
3,Australia,2019,2.403171e+12,2.194017,1.499348e+12,59181.299790,94856.424118,3.476600,3.874513e+10,2.770776,74.579592,20.289384,2.123277,54.290208,5.143,66.052,1.398350e+12
4,Australia,2018,2.351577e+12,2.874170,1.467158e+12,58772.706315,94201.526099,1.846176,6.068664e+10,4.234508,75.393593,19.942302,0.051922,55.451291,5.345,65.746,1.433145e+12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
675,United States,1987,9.627608e+12,3.454630,8.849219e+12,36523.404486,39736.051302,2.477388,6.323500e+10,1.302414,79.361511,16.001207,-2.981742,63.360304,6.200,65.600,4.855215e+12
676,United States,1986,9.306116e+12,3.462655,8.553720e+12,35620.759359,38754.008320,2.013893,3.094600e+10,0.675731,79.138843,16.114595,-2.879468,63.024248,7.000,65.300,4.579631e+12
677,United States,1985,8.994662e+12,4.169575,8.267447e+12,34748.266874,37804.770246,3.162525,9.630000e+09,0.221942,78.439421,15.917109,-2.627761,62.522312,7.200,64.800,4.338979e+12
678,United States,1984,8.634635e+12,7.236453,7.936527e+12,33654.307930,36614.585231,3.607879,2.523000e+10,0.624874,77.448433,15.720377,-2.544251,61.728056,7.500,64.400,4.037613e+12


#Importar modelo

In [194]:
def root_mean_squared_error(y_true, y_pred):
    import tensorflow as tf
    return tf.sqrt(tf.reduce_mean(tf.square(y_pred - y_true)))

modelow = load_model(
    ruta+"modelo_lstm_sintc.keras",
    custom_objects={"root_mean_squared_error": root_mean_squared_error}
)

#Importar parametros

In [195]:
with open(ruta+"conf_sint.json") as f:
    config = json.load(f)

input_length = config["input_length"]
output_length = config["output_length"]

##Variable que escojo para predecir

In [196]:
vars_keep = ["GDP_cnstn_usd"]

##Creo arrays

In [197]:
countries = df["country"].unique()
print(countries.shape)
countries

(17,)


array(['Australia', 'Austria', 'Belgium', 'Canada', 'Germany', 'Denmark',
       'Spain', 'France', 'United Kingdom', 'Greece',
       'Hong Kong SAR, China', 'Ireland', 'Italy', 'Japan', 'Korea, Rep.',
       'Norway', 'United States'], dtype=object)

In [198]:
#Datos
x_total_train, y_total_train, x_total_valid, y_total_valid, x_total_test, y_total_test =preparar_pipeline(vars_keep,input_length,output_length,vars_keep[0])

##Fine tuning

In [199]:
# Ajustar parámetros para reproducibilidad del entrenamiento
tf.random.set_seed(159)
tf.config.experimental.enable_op_determinism()

###Capas que voy a reentrenar

In [200]:
for layer in modelow.layers[:-2]:
    layer.trainable = False

In [201]:
for layer in modelow.layers:
    print(layer.name, layer.trainable)

lstm_8 False
lstm_9 False
dropout_4 False
dense_9 True
dense_10 True


###Reentreno del modelo

In [202]:
learning_rate=5e-5

In [203]:
optimizador=RMSprop(learning_rate=learning_rate)

In [204]:
modelow.compile(
    optimizer=optimizador,
    loss=root_mean_squared_error
)

In [205]:
BATCH_SIZE=50
EPOCHS=3000

In [206]:
# -------- EARLY STOP --------
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

In [207]:
# continuar entrenamiento
historia = modelow.fit(
    x=x_total_train,
    y=y_total_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(x_total_valid, y_total_valid),
    callbacks=[early_stop],
    verbose=2
)

Epoch 1/3000
6/6 - 3s - 471ms/step - loss: 0.2199 - val_loss: 0.2972
Epoch 2/3000
6/6 - 0s - 32ms/step - loss: 0.2176 - val_loss: 0.2949
Epoch 3/3000
6/6 - 0s - 37ms/step - loss: 0.2163 - val_loss: 0.2928
Epoch 4/3000
6/6 - 0s - 33ms/step - loss: 0.2147 - val_loss: 0.2909
Epoch 5/3000
6/6 - 0s - 33ms/step - loss: 0.2112 - val_loss: 0.2891
Epoch 6/3000
6/6 - 0s - 32ms/step - loss: 0.2104 - val_loss: 0.2875
Epoch 7/3000
6/6 - 0s - 52ms/step - loss: 0.2119 - val_loss: 0.2859
Epoch 8/3000
6/6 - 0s - 37ms/step - loss: 0.2092 - val_loss: 0.2842
Epoch 9/3000
6/6 - 0s - 33ms/step - loss: 0.2030 - val_loss: 0.2827
Epoch 10/3000
6/6 - 0s - 59ms/step - loss: 0.2060 - val_loss: 0.2813
Epoch 11/3000
6/6 - 0s - 71ms/step - loss: 0.2056 - val_loss: 0.2798
Epoch 12/3000
6/6 - 0s - 53ms/step - loss: 0.2052 - val_loss: 0.2786
Epoch 13/3000
6/6 - 0s - 54ms/step - loss: 0.2055 - val_loss: 0.2772
Epoch 14/3000
6/6 - 0s - 69ms/step - loss: 0.1966 - val_loss: 0.2758
Epoch 15/3000
6/6 - 0s - 51ms/step - loss:

In [208]:
# Cálculo de rmses para train, val y test
rmse_tr = modelow.evaluate(x=x_total_train, y=y_total_train, verbose=0)
rmse_vl = modelow.evaluate(x=x_total_valid, y=y_total_valid, verbose=0)
rmse_ts = modelow.evaluate(x=x_total_test, y=y_total_test, verbose=0)

# Imprimir resultados en pantalla
print('\nComparativo desempeños:')
print(f'  RMSE train:\t {rmse_tr:.3f}')
print(f'  RMSE val:\t {rmse_vl:.3f}')
print(f'  RMSE test:\t {rmse_ts:.3f}')



Comparativo desempeños:
  RMSE train:	 0.155
  RMSE val:	 0.246
  RMSE test:	 0.275


#Guardar modelo

In [210]:
modelow.save(ruta+"modelo_fine_tuning.keras")